# Ejercicio 5 — Programación Orientada a Objetos
## Registro Climático de Guatemala — Examen Final (Variante B)

**Nombre:** Mia Zoé Gutiérrez Marroquín  
**Carnet:** 22600433

---

Este notebook usa las clases que implementaste en `poo_clima.py`.

**Requisito previo:** haber completado todos los métodos de `poo_clima.py`.

**Entrega:**
- `poo_clima.py` con tu implementación
- Este notebook ejecutado con todos los outputs visibles

> **Ruta esperada en tu repositorio:**  
> `examenes/python/examen_final/poo_clima.py`  
> `examenes/python/examen_final/examen_final_poo_b.ipynb`

In [21]:
import pandas as pd
import requests

URL_CLIMA = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=14.64&longitude=-90.51"
    "&start_date=2024-01-01&end_date=2024-12-31"
    "&daily=temperature_2m_max,temperature_2m_min"
    ",precipitation_sum,wind_speed_10m_max"
    "&timezone=America%2FGuatemala"
)

respuesta = requests.get(URL_CLIMA, timeout=15)
clima = pd.DataFrame(respuesta.json()['daily'])
clima['time'] = pd.to_datetime(clima['time'])
print(f"Dataset cargado: {len(clima)} días")
clima.head(3)

Dataset cargado: 366 días


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max
0,2024-01-01,23.8,10.7,0.0,15.9
1,2024-01-02,24.1,13.1,0.2,18.0
2,2024-01-03,27.5,12.8,0.1,18.5


In [22]:
# Reutiliza la función clasificar_dia() del Ejercicio 3 para agregar la columna 'tipo_dia'
def clasificar_dia(precip):
    if precip < 1:
        return 'Seco'
    elif precip < 5:
        return 'Lluvia ligera'
    elif precip < 20:
        return 'Lluvia moderada'
    else:
        return 'Lluvia intensa'

clima['tipo_dia'] = clima['precipitation_sum'].apply(clasificar_dia)
print("Columna 'tipo_dia' agregada.")
print(clima['tipo_dia'].value_counts())

Columna 'tipo_dia' agregada.
tipo_dia
Seco               203
Lluvia ligera       88
Lluvia moderada     59
Lluvia intensa      16
Name: count, dtype: int64


In [23]:
from poo_clima import DiaClimatico, RegistroAnual
print("Clases importadas correctamente.")

Clases importadas correctamente.


---
## Parte 1 — Probando la clase `DiaClimatico` (objetos individuales)

El siguiente bloque crea tres objetos `DiaClimatico` representativos y llama a todos sus métodos.  
Verifica que el output tiene sentido con los datos climáticos.

In [24]:
# Tres días representativos del DataFrame
fila_primero  = clima.iloc[0]
fila_caluroso = clima.loc[clima['temperature_2m_max'].idxmax()]
fila_lluvioso = clima.loc[clima['precipitation_sum'].idxmax()]

def crear_dia(fila):
    return DiaClimatico(
        fila['time'].date(),
        fila['temperature_2m_max'],
        fila['temperature_2m_min'],
        fila['precipitation_sum'],
        fila['wind_speed_10m_max'],
    )

d_primero  = crear_dia(fila_primero)
d_caluroso = crear_dia(fila_caluroso)
d_lluvioso = crear_dia(fila_lluvioso)

for etiqueta, d in [("Primero del año",       d_primero),
                    ("Día más caluroso",       d_caluroso),
                    ("Día más lluvioso",       d_lluvioso)]:
    print(f"=== {etiqueta} ===")
    print(f"  {d}")
    print(f"  clasificar()   → {d.clasificar()}")
    print(f"  rango_termico()→ {d.rango_termico():.1f} °C")
    print(f"  temp_media()   → {d.temp_media():.1f} °C")
    print(f"  es_caluroso()  → {d.es_caluroso()}")
    print()

=== Primero del año ===
  2024-01-01 | max=23.8°C  min=10.7°C | Seco | Viento: 15.9 km/h
  clasificar()   → Seco
  rango_termico()→ 13.1 °C
  temp_media()   → 17.2 °C
  es_caluroso()  → False

=== Día más caluroso ===
  2024-05-07 | max=33.3°C  min=15.1°C | Seco | Viento: 23.4 km/h
  clasificar()   → Seco
  rango_termico()→ 18.2 °C
  temp_media()   → 24.2 °C
  es_caluroso()  → True

=== Día más lluvioso ===
  2024-06-17 | max=19.4°C  min=16.4°C | Lluvia intensa | Viento: 33.7 km/h
  clasificar()   → Lluvia intensa
  rango_termico()→ 3.0 °C
  temp_media()   → 17.9 °C
  es_caluroso()  → False



### Tu turno — elige un día

Crea un objeto `DiaClimatico` usando **cualquier fila** del DataFrame que tú elijas  
(puede ser el día de tu cumpleaños, el más frío, el más ventoso, etc.).  
Muestra su `descripcion()` y justifica tu elección en un comentario.

In [32]:
# Elige una fila y crea tu propio objeto DiaClimatico
# Hint: clima[clima['time'].dt.month == 8].iloc[0]  → primer día de agosto
#       clima.loc[clima['wind_speed_10m_max'].idxmax()]  → día más ventoso

# TU CÓDIGO AQUÍ

#escogí el 26/02/2024 porque es la fecha de mi cumpleaños
fila = clima[clima['time'] == pd.to_datetime("2024-02-26")].iloc[0]
dia_cumple  = DiaClimatico(
    fila['time'].date(),
    fila['temperature_2m_max'],
    fila['temperature_2m_min'],
    fila['precipitation_sum'],
    fila['wind_speed_10m_max'],
)
print(dia_cumple)
print(f"¿Es caluroso? {dia_cumple.es_caluroso()}")
print(f"Rango térmico: {dia_cumple.rango_termico():.1f} °C")

2024-02-26 | max=26.1°C  min=11.4°C | Seco | Viento: 20.8 km/h
¿Es caluroso? False
Rango térmico: 14.7 °C


---
## Parte 2 — Construyendo el `RegistroAnual` con todos los datos

Crea el registro con los **366 días** del año 2024 y ejerce los métodos de consulta.

In [33]:
registro = RegistroAnual("Ciudad de Guatemala", 2024)

for indice, fila in clima.iterrows():
    registro.agregar_dia(DiaClimatico(
        fecha         = fila['time'].date(),
        temp_max      = fila['temperature_2m_max'],
        temp_min      = fila['temperature_2m_min'],
        precipitacion = fila['precipitation_sum'],
        viento_max    = fila['wind_speed_10m_max'],
    ))

print(f"Registro construido: {len(registro)} días")

Registro construido: 366 días


In [35]:
# ── dia_mas_caluroso() ───────────────────────────────────────────────────────
mas_caluroso = registro.dia_mas_caluroso()
print("Día más caluroso del año:")
print(" ", mas_caluroso)
print(f"  Rango térmico : {mas_caluroso.rango_termico():.1f} °C")
print(f"  Temperatura media: {mas_caluroso.temp_media():.1f} °C")

Día más caluroso del año:
  2024-05-07 | max=33.3°C  min=15.1°C | Seco | Viento: 23.4 km/h
  Rango térmico : 18.2 °C
  Temperatura media: 24.2 °C


In [36]:
# ── dias_por_tipo() ──────────────────────────────────────────────────────────
print("Días por tipo (año completo):")
for tipo in ['Seco', 'Lluvia ligera', 'Lluvia moderada', 'Lluvia intensa']:
    grupo = registro.dias_por_tipo(tipo)
    print(f"  {tipo:<18}: {len(grupo)} días")
    for d in grupo[:2]:
        print(f"      {d}")

Días por tipo (año completo):
  Seco              : 203 días
      2024-01-01 | max=23.8°C  min=10.7°C | Seco | Viento: 15.9 km/h
      2024-01-02 | max=24.1°C  min=13.1°C | Seco | Viento: 18.0 km/h
  Lluvia ligera     : 88 días
      2024-01-20 | max=24.5°C  min=15.4°C | Lluvia ligera | Viento: 26.8 km/h
      2024-01-28 | max=23.1°C  min=15.6°C | Lluvia ligera | Viento: 31.4 km/h
  Lluvia moderada   : 59 días
      2024-02-18 | max=27.0°C  min=13.3°C | Lluvia moderada | Viento: 17.9 km/h
      2024-04-04 | max=27.0°C  min=16.2°C | Lluvia moderada | Viento: 14.6 km/h
  Lluvia intensa    : 16 días
      2024-06-15 | max=23.8°C  min=16.9°C | Lluvia intensa | Viento: 16.6 km/h
      2024-06-17 | max=19.4°C  min=16.4°C | Lluvia intensa | Viento: 33.7 km/h


In [37]:
# ── temp_promedio_anual() + resumen() ────────────────────────────────────────
print(f"Temperatura máxima promedio anual: {registro.temp_promedio_anual()} °C\n")
registro.resumen()

Temperatura máxima promedio anual: 25.8 °C

Registro anual de Ciudad de Guatemala - 2024
Total de días registrados: 366
Día más caluroso: 2024-05-07 con 33.3°C
Temperatura máxima promedio: 25.8°C
Días de tipo 'Seco': 203
Días de tipo 'Lluvia ligera': 88
Días de tipo 'Lluvia moderada': 59
Días de tipo 'Lluvia intensa': 16


### Tu turno — días calurosos y lluviosos

Usa `dias_por_tipo('Lluvia intensa')` para obtener los días con lluvia intensa  
y determina cuántos de ellos también fueron calurosos (`es_caluroso() == True`).  
Muestra el porcentaje.

In [43]:
# Hint: obtén la lista de días de lluvia intensa con dias_por_tipo()
#       luego cuenta cuántos tienen es_caluroso() == True con un for

# TU CÓDIGO AQUÍ
print("Días de lluvia intensa que también fueron calurosos:")
intensos_calurosos = 0
for dia in registro.dias_por_tipo("Lluvia intensa"):
    if dia.es_caluroso () == True:
        print(f"  {dia}")
        intensos_calurosos += 1
porcentaje = (intensos_calurosos / len(registro.dias_por_tipo("Lluvia intensa"))) * 100
print(f"Total: {intensos_calurosos} días ({porcentaje:.1f}%)")


Días de lluvia intensa que también fueron calurosos:
Total: 0 días (0.0%)


---
## Parte 3 — Verificación automática

Esta celda confirma que tu `clasificar()` produce los mismos resultados  
que la función `clasificar_dia()` del Ejercicio 3 para **todos** los días.

In [44]:
errores = 0
for _, fila in clima.iterrows():
    d = DiaClimatico(fila['time'].date(), fila['temperature_2m_max'],
                     fila['temperature_2m_min'], fila['precipitation_sum'],
                     fila['wind_speed_10m_max'])
    if d.clasificar() != fila['tipo_dia']:
        errores += 1
        print(f"  ✗ {fila['time'].date()}  precip={fila['precipitation_sum']:.1f}"
              f"  pandas={fila['tipo_dia']}  poo={d.clasificar()}")

if errores == 0:
    print(f"✓ Todos los {len(clima)} días clasifican correctamente.")
else:
    print(f"✗ {errores} días con clasificación incorrecta. Revisa tu método clasificar().")

✓ Todos los 366 días clasifican correctamente.


---
## Preguntas de interpretación

1. ¿En qué fecha fue el día más caluroso del año? ¿En qué mes cae?  
   ¿Coincide con lo que esperarías del clima de Guatemala? Justifica.

   *Tu respuesta:* La fecha más calurosa del se presentó el 2024/05/07 justo en el mes de mayo con una temperatura de 33.3°C. Esta fecha coincide con lo que se esperaría ya que el mes que presentó mayor temperatura fue mayo con un promedio de 30°C aproximadamente.

2. ¿Cuántos días de `'Lluvia intensa'` también fueron calurosos (`es_caluroso() == True`)?  
   ¿Qué te dice eso sobre la relación entre temperatura y lluvia intensa en Guatemala?

   *Tu respuesta:* No hubo ningún día que fuera catalogado como caluroso y que haya tenido lluvia intensa, eso indica que a mayor cantidad de lluvia menor temperatura y viseversa. 

3. ¿Qué ventaja tiene usar la clase `RegistroAnual` frente a trabajar directamente  
   con el DataFrame de pandas para este tipo de consultas?

   *Tu respuesta:* Una de las ventajas de trabajar con RegistroAnual es que se puede tener código más limpio, además es útil ya que se puede reutilizar el código por lo que no se encontrará repetido a lo largo del trabajo y es posible de utilizarlo para análisis similares.